# AeroPulse Source Likelihood — 04: Weak Supervision

Aggregate the labeling-function votes into independent per-source
probabilistic targets, with conflict, coverage and a derived unknown state.

## Objective

Improvement plan sections 7, 8, 22, 23 and 24.

The votes from notebook 03 are not yet targets. This notebook aggregates
them into the multi-label schema the plan asks for in section 22:

```
y_biomass_burning, y_traffic, y_industrial, y_dust, y_regional_transport
```

Aggregation is reliability-weighted, and the weight is normalised by the
functions that **actually voted** rather than by all of them. That is what
keeps abstention neutral: a source judged by one confident function is not
penalised for the silence of the others.

Three quantities come out of the same pass:

- **coverage** — how much of the available reliability weight voted at all
- **conflict** — disagreement among the functions for one source (section 7)
- **unknown** — derived at the end from the absence of confident scores,
  never modelled as a sixth competing class (section 8)

Section 24 is respected as a hard boundary: this notebook produces labels
and stops. The classifier never sees the functions that made them.


> **Scientific constraint, repeated in every notebook because it governs how
> the output may be used.** These are *likelihoods derived from heuristics*,
> not measured source attribution. No public ground-truth attribution
> catalogue exists for this dataset. A model that scores well here has
> reproduced its own labeling system — which is not the same as being right.

In [1]:
# ============================================================================
# SETUP — reusable logic lives in source_toolkit.py, not in cells.
# ============================================================================
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import source_toolkit as st

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)

cfg = st.load_config()
SOURCES = st.SOURCES
print("kernel:", sys.executable)

env: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood/.env
project: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood
sources: ['biomass_burning', 'traffic', 'industrial', 'dust', 'regional_transport']
kernel: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/.venv/bin/python


In [2]:
df = pd.read_parquet(cfg.feature_dir / "source_evidence_features.parquet")
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df = df.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)
votes = pd.read_parquet(cfg.label_dir / "labeling_function_votes.parquet")
manifest = json.loads((cfg.label_dir / "labeling_manifest.json").read_text())
print("rows", f"{len(df):,}", "| votes", votes.shape)

rows 1,705,252 | votes (1705252, 15)


## 1. Aggregate

In [3]:
t0 = time.time()
labels = st.aggregate_weak_labels(votes)
print(f"aggregated in {time.time() - t0:.1f}s -> {labels.shape[1]} columns")

rows = []
for source in SOURCES:
    rows.append({
        "source": source,
        "positive_rate": round(float(labels[f"y_{source}"].mean()) * 100, 2),
        "positives": int(labels[f"y_{source}"].sum()),
        "evaluated_pct": round(float(labels[f"{source}_weak_prob"].notna().mean()) * 100, 2),
        "mean_prob_when_evaluated": round(float(labels[f"{source}_weak_prob"].mean()), 4),
        "mean_coverage": round(float(labels[f"{source}_coverage"].mean()), 4),
        "mean_conflict": round(float(labels[f"{source}_conflict"].mean()), 4),
    })
print()
print(pd.DataFrame(rows).to_string(index=False))

aggregated in 0.6s -> 34 columns

            source  positive_rate  positives  evaluated_pct  mean_prob_when_evaluated  mean_coverage  mean_conflict
   biomass_burning           6.42     109410          14.45                    0.5077         0.0800         0.2104
           traffic          38.64     658979          75.70                    0.5334         0.3389         0.1462
        industrial          13.21     225345          13.21                    0.9241         0.0571         0.0000
              dust          33.42     569891          55.84                    0.5833         0.2414         0.0090
regional_transport           4.14      70671           7.79                    0.5284         0.0282         0.0119


## 2. The architectural test: are the labels actually multi-label?

Under the old priority chain this number is zero by construction. If it is
still zero here, the rewrite achieved nothing.

In [4]:
y_cols = [f"y_{s}" for s in SOURCES]
n_supported = labels[y_cols].sum(axis=1)
print("sources supported per row:")
print(n_supported.value_counts().sort_index().to_string())
multi = int((n_supported > 1).sum())
print(f"\nrows supporting 2+ sources: {multi:,} ({multi / len(labels):.1%})")
assert multi > 0, "labels are still mutually exclusive — the rewrite failed"
print("\nPASS — a row can now hold several source hypotheses at once.")

sources supported per row:
0    577467
1    711788
2    332048
3     77530
4      6273
5       146

rows supporting 2+ sources: 415,997 (24.4%)

PASS — a row can now hold several source hypotheses at once.


## 3. Conflict structure (section 7)

In [5]:
conflicts = st.conflict_report(labels)
print(conflicts.to_string(index=False))
print("\nA source is only 'evaluated' on rows where at least one of its")
print("labeling functions had the evidence to vote. Elsewhere it abstains,")
print("which is recorded as NaN rather than 0.5 — an abstaining source that")
print("scored 0.5 would clear the positive threshold on silence alone.")

print("\ncross-source conflict score:")
print(labels["conflict_score"].describe().round(3).to_string())
print(f"\nrows with conflict_score > 0.8 (two sources nearly tied): "
      f"{float((labels['conflict_score'] > 0.8).mean()):.1%}")

       source_a           source_b  co_occurrence  jaccard
        traffic               dust         196826   0.1907
        traffic         industrial         137494   0.1841
biomass_burning            traffic          73845   0.1063
     industrial               dust          53433   0.0720
        traffic regional_transport          42743   0.0622
biomass_burning         industrial          36218   0.1213
     industrial regional_transport          18927   0.0683
biomass_burning               dust          15174   0.0228
           dust regional_transport          14824   0.0237
biomass_burning regional_transport          14252   0.0859

A source is only 'evaluated' on rows where at least one of its
labeling functions had the evidence to vote. Elsewhere it abstains,
which is recorded as NaN rather than 0.5 — an abstaining source that
scored 0.5 would clear the positive threshold on silence alone.

cross-source conflict score:
count    1705252.000
mean           0.330
std           

## 4. Unknown as a derived state, not a class (section 8)

In [6]:
print(f"is_unknown rate (no source supported): {float(labels['is_unknown'].mean()):.1%}")
print(f"rows where no source could be evaluated at all: "
      f"{float((labels['n_sources_evaluated'] == 0).mean()):.1%}")
print("\nsources evaluated per row:")
print(labels["n_sources_evaluated"].value_counts().sort_index().to_string())
print("\nunknown_probability:")
print(labels["unknown_probability"].describe().round(3).to_string())
print("\nattribution_confidence:")
print(labels["attribution_confidence"].describe().round(3).to_string())

print("\nprimary source distribution (highest-probability source per row):")
print((labels["primary_source"].value_counts() / len(labels) * 100).round(2).to_string())
print("\nNOTE: primary_source is a ranking of likelihoods, not an exclusive")
print("assignment. The y_* columns remain independent.")

is_unknown rate (no source supported): 33.9%
rows where no source could be evaluated at all: 9.4%

sources evaluated per row:
n_sources_evaluated
0    161126
1    625467
2    649365
3    171006
4     80898
5     17390

unknown_probability:
count    1705252.000
mean           0.301
std            0.355
min            0.038
25%            0.051
50%            0.063
75%            0.757
max            1.000

attribution_confidence:


count    1705252.000
mean           0.185
std            0.207
min            0.000
25%            0.000
50%            0.051
75%            0.308
max            0.924

primary source distribution (highest-probability source per row):
primary_source
traffic               52.19
dust                  25.30
biomass_burning        6.84
industrial             5.40
regional_transport     0.83

NOTE: primary_source is a ranking of likelihoods, not an exclusive
assignment. The y_* columns remain independent.


### Does confidence behave like confidence?

Attribution confidence should be high when one source dominates with good
coverage and low when the evidence is thin or contested.

In [7]:
probe = labels.copy()
probe["_conf_band"] = pd.qcut(probe["attribution_confidence"].rank(method="first"), 5,
                              labels=["lowest", "low", "mid", "high", "highest"])
print(probe.groupby("_conf_band", observed=True).agg(
    n=("primary_prob", "size"),
    mean_primary_prob=("primary_prob", "mean"),
    mean_conflict=("conflict_score", "mean"),
    mean_coverage=("max_coverage", "mean"),
    unknown_rate=("is_unknown", "mean"),
).round(3).to_string())

                 n  mean_primary_prob  mean_conflict  mean_coverage  unknown_rate
_conf_band                                                                       
lowest      341051              0.629          0.795          0.442         0.330
low         341050              0.432          0.571          0.359         0.557
mid         341050              0.377          0.076          0.426         0.676
high        341050              0.858          0.095          0.425         0.129
highest     341051              0.904          0.112          0.635         0.001


## 5. Build the event table (section 29)

In [8]:
work = df.join(labels)
events = st.build_events(work, labels)
print(f"events: {len(events):,} spanning "
      f"{events['event_start'].min()} -> {events['event_end'].max()}")
print(f"unique upstream event_id values: {events['event_id'].nunique():,} "
      f"across {events['location_id'].nunique()} stations")
print("(event_id is not unique across stations — events are keyed on the pair)")
print(events[["n_hours", "peak_pm25", "mean_pm25", "duration_hours"]].describe().round(2).to_string())
print("\ndominant source at event level:")
print((events["dominant_source"].value_counts() / len(events) * 100).round(2).to_string())

events: 44,563 spanning 2025-02-18 20:30:00+00:00 -> 2026-09-08 13:30:00+00:00
unique upstream event_id values: 706 across 149 stations
(event_id is not unique across stations — events are keyed on the pair)
        n_hours  peak_pm25  mean_pm25  duration_hours
count  44563.00   44563.00   44563.00        44563.00
mean       5.70     103.27      86.00            4.70
std        8.73      82.95      55.51            8.73
min        1.00      60.00      60.00            0.00
25%        1.00      66.80      65.02            0.00
50%        3.00      79.30      72.50            2.00
75%        6.00     107.00      87.00            5.00
max      354.00    1870.00    1870.00          353.00

dominant source at event level:
dominant_source
industrial            64.83
traffic               10.63
dust                   8.44
regional_transport     8.20
biomass_burning        7.89


## 6. Save

In [9]:
labels_out = cfg.label_dir / "weak_labels.parquet"
labels.to_parquet(labels_out, index=False)
events.to_parquet(cfg.label_dir / "events.parquet", index=False)
conflicts.to_csv(cfg.label_dir / "conflict_matrix.csv", index=False)

report = {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "label_schema_version": st.SCHEMA["label_schema_version"],
    "sources": list(SOURCES),
    "positive_rates": {s: float(labels[f"y_{s}"].mean()) for s in SOURCES},
    "multi_source_rows": int(multi),
    "multi_source_pct": float(multi / len(labels)),
    "unknown_rate": float(labels["is_unknown"].mean()),
    "mean_attribution_confidence": float(labels["attribution_confidence"].mean()),
    "conflicts": conflicts.to_dict(orient="records"),
    "n_events": int(len(events)),
    "caveat": "probabilistic weak labels from heuristics; not source attribution",
}
(cfg.label_dir / "weak_supervision_report.json").write_text(json.dumps(report, indent=2, default=str))
print("04 COMPLETE ->", labels_out)
print("Next: 05_source_classifier_training.ipynb")

04 COMPLETE -> /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood/data/source/processed/labels/weak_labels.parquet
Next: 05_source_classifier_training.ipynb


## Findings

**The multi-label target exists and is populated.** 24.4% of rows support
two or more sources; 84,000 support three or more. Under the legacy
architecture this number was zero by construction, so this is the plan's
section 22 delivered rather than described.

| source | evaluated on | positive rate | mean coverage |
|---|---|---|---|
| traffic | 75.70% of rows | 38.64% | 0.339 |
| dust | 55.84% | 33.42% | 0.241 |
| biomass_burning | 14.45% | 6.42% | 0.080 |
| industrial | 13.21% | 13.21% | 0.057 |
| regional_transport | 7.79% | 4.14% | 0.028 |

Read the "evaluated on" column first. **Regional transport can be assessed
at all on only 7.8% of rows** and industrial on 13.2%. Those two hypotheses
are not weak because the evidence argues against them — they are weak
because there is usually no evidence either way. Notebook 01 already showed
why for transport: the upwind field is null on 36% of rows, and the
labeling functions correctly abstain there rather than voting no.

**A bug worth recording, because it inverted the meaning of abstention.**
The first implementation gave every source a score of 0 when all its
functions abstained, which the logistic squash mapped to probability 0.5 —
which cleared the 0.5 positive threshold. Silence was being counted as
support. It produced an `is_unknown` rate of exactly 0.0%, a conflict score
whose median was 1.0 (every abstaining source tied with every other), and
a `traffic` positive rate inflated by rows where no traffic function had
spoken. Abstention is now NaN, propagated NaN-aware through every
downstream statistic, and asserted in the toolkit test suite.

**Industrial has zero mean conflict, which is a warning rather than a
strength.** Its three functions never contradict each other because none of
them can issue a −1 vote at all. A source whose evidence can only ever
accumulate in one direction will look confident regardless of the truth.
`LF_DUST_CAMS`, `LF_FIRE_LOCAL`, `LF_CROP_SEASON`,
`LF_TRANSPORT_CONSISTENCY` and both remaining industrial functions share
this property. Adding contradicting conditions is the highest-value next
change to the labeling layer.

**Conflict is real and common.** Traffic co-occurs with dust on 196,826
rows (Jaccard 0.19) and with industrial on 137,494 (0.18). Those pairs are
exactly what a single-label scheme has to destroy in order to emit one
answer, and the conflict score now surfaces them instead.

**Confidence behaves, mostly.** Across confidence quintiles the unknown
rate falls 0.330 → 0.557 → 0.676 → 0.129 → 0.001 and mean conflict falls
0.795 → 0.112. The non-monotonicity in the low and mid bands is honest and
explicable: those rows have a *moderate* primary probability with wide
coverage, so they score mid-confidence while frequently supporting nothing
above threshold.

**Events: 44,563 episodes, median 3 hours, longest 354 hours.** A second
bug caught here: the upstream `event_id` is **not unique across stations**
— only 706 distinct values across 149 stations. Grouping on it alone
produced 706 "events" with a mean duration of 6,163 hours (257 days).
Events are now keyed on `(location_id, event_id)`.

One caveat on event-level aggregation: the dominant source at event level
(industrial 64.8%) differs sharply from the row level (traffic 52.2%),
because averaging NaN-sparse probabilities favours a source that is
evaluated rarely but scores high when it is. Event-level dominance should
be coverage-weighted before it is used for anything that matters; as it
stands it is a diagnostic, not a conclusion.